Function Calling

In [65]:
from dotenv import load_dotenv
import os

load_dotenv()

# 사용할 키 가져오기
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [66]:
# Open AI API 연결
from openai import OpenAI

In [67]:
# 연결된 open ai 객체 생성
client = OpenAI(api_key=OPENAI_API_KEY)

In [68]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [69]:
tools = [
    {
        'type' : 'function',
        'name' : 'get_weather',
        'description' : 'Get current temperature for provided coordinates in celsius.',
        'parameters' : {
            'type' : 'object',
            'properties' : {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
        'required' : ["latitude", "longitude"],
        'additionalProperties' : False
        },
        'strict' : True
    }
]

In [70]:
input_message = [
    {
        "role" : "user",
        "content" : "오늘 파리 날씨는 어때?"
    }
]

In [71]:
response = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [72]:
response.output[1]

ResponseFunctionToolCall(arguments='{"latitude":48.8566,"longitude":2.3522}', call_id='call_B9SW8Zl7c7OiEpSoOoX4tQum', name='get_weather', type='function_call', id='fc_0a7393c2ab6506e0006a7585df06b4819bb704a46db6f31726', caller=None, namespace=None, status='completed')

In [73]:
from pprint import pprint

for item in response.output:
    d_temp = dict(item)
    pprint(d_temp, indent=2, width=50)            # indent : 들여쓰기

{ 'content': [],
  'encrypted_content': 'gAAAAABqdYXnFx0NWW4cQpgpG2NRteIEsv6ngocdPdGkxSPVczx2uc_dMDqlQYnela9pilByfTeH0I0NmPt22m4Z67W1TlzABjxTvZRXTnwi52T6upNKitjYz0xmz2dA9YJFquN6291eBq5uaQi2kbwzcHlXk6_Q7M9hIINijF7QPUsG79Z9krxQUE-5ii1YCurRpEkfVhF8LMebRNhwe-gz9r30k386dYD_CvP3iqtPJhnXhVd4DtsQSBnIjCPPjEATnOXD1Pp7SyTddIbZPijfyfKeQnEoQtsZs2I93MbRqmfc4JbGhfwmuewxrPadhmIT3nZ_KwC3VmyxlvBTy488EXH0DyzR9vzWmgFoMdvlrFYk8Blpk5NFhe9NDS9fxNx0VOG9GS2LUPQ-fk4WWv0em9ygf_MRoC9rGpTCJldWMqwa8C5WAqL2nbwLmYKRVlJ20Nzi31V_HoTOcSUl5VxS8pTq6xiZ2RnNZpKeK1LVV9Fp0r8lXpXnDlfeSwXcHTA4bmiFM5UC1zKrT_KQKteJnnaJFhKZjBPTsB-hD4SeQ7-6xA3UpEjZdSr_6tqHktrDpl_2QM4UEXCHp15vqYlF5c7q0F8LN5SsoWtaTrZu16L_TKWaSf5Mrc8A4HPDp4kJ4gvUi2mSyNZPn81CIALqo4djHweepQ8GNkUEwcNxBgamgsiAONkV4Lm3f4YNGShxmLuk5xgwO6k_pda6oPt1yDIjjeEwuLLq9q3SsCEshqDpuPU_wIUYdN-XZpYl-SfexYiTvW7le6Y2rAj3c9Ygm-Ra-C2_fkbcDletf1ZRm_15JNg1Zd8bCJN6ptNWBSREn0iaHg8h1_hwhzIup_pVCj_n3Em4-CTokp3h0jlrYE5Eh-CQwXoo2e3U7bkciUysXZS-yl0HrCha-bTPz__npkh-6CKDyLdwFfwn9OeTq5CxkOJloSo1-65sX3u

In [74]:
# 함수 호출에 필요한 파라미터 추출
import json

tool_call = response.output[1]

# json -> object
args = json.loads(tool_call.arguments)

print(args)

{'latitude': 48.8566, 'longitude': 2.3522}


In [75]:
# 함수 호출
print(tool_call.name)

result = get_weather(args["latitude"], args["longitude"])

print("result : ", result)

get_weather
result :  16.0


In [76]:
# 메시지 병합
# 순서 : 요청 -> 응답 : function_call -> 함수 실행
input_message += response.output

# 함수의 실행 결과도 메시지에 병합

input_message.append(
    {
        "type" : "function_call_output",
        "call_id" : tool_call.call_id,
        "output" : str(result)
    }
)

print(input_message)

[{'role': 'user', 'content': '오늘 파리 날씨는 어때?'}, ResponseReasoningItem(id='rs_0a7393c2ab6506e0006a7585deb194819bb68a82015be37bc2', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqdYXnFx0NWW4cQpgpG2NRteIEsv6ngocdPdGkxSPVczx2uc_dMDqlQYnela9pilByfTeH0I0NmPt22m4Z67W1TlzABjxTvZRXTnwi52T6upNKitjYz0xmz2dA9YJFquN6291eBq5uaQi2kbwzcHlXk6_Q7M9hIINijF7QPUsG79Z9krxQUE-5ii1YCurRpEkfVhF8LMebRNhwe-gz9r30k386dYD_CvP3iqtPJhnXhVd4DtsQSBnIjCPPjEATnOXD1Pp7SyTddIbZPijfyfKeQnEoQtsZs2I93MbRqmfc4JbGhfwmuewxrPadhmIT3nZ_KwC3VmyxlvBTy488EXH0DyzR9vzWmgFoMdvlrFYk8Blpk5NFhe9NDS9fxNx0VOG9GS2LUPQ-fk4WWv0em9ygf_MRoC9rGpTCJldWMqwa8C5WAqL2nbwLmYKRVlJ20Nzi31V_HoTOcSUl5VxS8pTq6xiZ2RnNZpKeK1LVV9Fp0r8lXpXnDlfeSwXcHTA4bmiFM5UC1zKrT_KQKteJnnaJFhKZjBPTsB-hD4SeQ7-6xA3UpEjZdSr_6tqHktrDpl_2QM4UEXCHp15vqYlF5c7q0F8LN5SsoWtaTrZu16L_TKWaSf5Mrc8A4HPDp4kJ4gvUi2mSyNZPn81CIALqo4djHweepQ8GNkUEwcNxBgamgsiAONkV4Lm3f4YNGShxmLuk5xgwO6k_pda6oPt1yDIjjeEwuLLq9q3SsCEshqDpuPU_wIUYdN-XZpYl-SfexYiTvW7le6Y2rAj3c9Ygm-Ra-C2_fkbcDletf1

In [77]:
# 최종 결과 요청
response2 = client.responses.create(
    model="gpt-5.5",
    input=input_message,
    tools=tools
)

In [78]:
print(response2.output)

[ResponseOutputMessage(id='msg_0a7393c2ab6506e0006a7585ea3674819ba05ad4514abd6956', content=[ResponseOutputText(annotations=[], text='오늘 파리의 현재 기온은 약 **16°C**입니다.  \n가벼운 겉옷을 챙기면 좋을 날씨예요.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [79]:
print(response2.output_text)

오늘 파리의 현재 기온은 약 **16°C**입니다.  
가벼운 겉옷을 챙기면 좋을 날씨예요.
